In [14]:
import pandas as pd
import numpy as np

In [15]:
yellow_taxi_jan_2026 = pd.read_parquet("../cleaned_data/clean_yellow_tripdata_2026-01.parquet")
# yellow_taxi_jan_2026 = pd.read_parquet("../raw_data/yellow_tripdata_2026-01.parquet")

In [16]:
yellow_taxi_jan_2026.sample(4)

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,vendor_name
2404204,2,2026-01-29 16:01:58,2026-01-29 16:07:50,1.0,0.35,1.0,N,263,236,2,...,0.00,0.5,0.0,0.00,1.0,10.50,2.5,0.00,0.00,"Curb Mobility, LLC"
633155,1,2026-01-08 23:18:26,2026-01-08 23:27:53,1.0,2.10,1.0,N,229,142,2,...,4.25,0.5,0.0,0.00,1.0,16.45,2.5,0.00,0.75,"Creative Mobile Technologies, LLC"
807986,2,2026-01-10 19:33:00,2026-01-10 19:55:11,2.0,7.21,5.0,N,138,107,1,...,5.00,0.0,1.0,7.46,1.0,79.46,2.5,1.75,0.75,"Curb Mobility, LLC"
1600691,2,2026-01-19 16:03:32,2026-01-19 16:07:06,1.0,0.58,1.0,N,162,170,1,...,2.50,0.5,1.5,0.00,1.0,14.55,2.5,0.00,0.75,"Curb Mobility, LLC"


In [17]:
df = yellow_taxi_jan_2026.copy()

# normalize the col name 

In [18]:
df.rename(columns={
    'tpep_pickup_datetime' : 'pickup_datetime',
    'tpep_dropoff_datetime' : 'dropoff_datetime'
}, inplace=True)

# check pickup time < dropoff time

In [19]:
df = df[(df['pickup_datetime'] < df['dropoff_datetime'])]

In [20]:
df.shape

(3679819, 21)

# Check dates, not included in 2026 jan. 2025-12-31 and 2026-02-01 is not midnight trips

In [21]:
print(df[
    (df['pickup_datetime'] >= '2025-12-31') |
    (df['dropoff_datetime'] <= '2026-02-01')
].shape)
df = df[
    (df['pickup_datetime'] >= '2025-12-31') |
    (df['dropoff_datetime'] <= '2026-02-01')
]


(3679819, 21)


# calculate trip duration

In [22]:
df['trip_duration'] = df['dropoff_datetime'] - df['pickup_datetime']

# remove trips having trip duration less than 2 min

In [23]:
df = df[(df['trip_duration'] >= pd.Timedelta(minutes=2))]

In [24]:
df.shape

(3618604, 22)

In [25]:
df.info()

<class 'pandas.DataFrame'>
Index: 3618604 entries, 0 to 3724888
Data columns (total 22 columns):
 #   Column                 Dtype          
---  ------                 -----          
 0   vendor_id              int8           
 1   pickup_datetime        datetime64[us] 
 2   dropoff_datetime       datetime64[us] 
 3   passenger_count        float64        
 4   trip_distance          float64        
 5   RatecodeID             float64        
 6   store_and_fwd_flag     str            
 7   PULocationID           int32          
 8   DOLocationID           int32          
 9   payment_type           int64          
 10  fare_amount            float64        
 11  extra                  float64        
 12  mta_tax                float64        
 13  tip_amount             float64        
 14  tolls_amount           float64        
 15  improvement_surcharge  float64        
 16  total_amount           float64        
 17  congestion_surcharge   float64        
 18  Airport_fee       

# save back to the cleaned file 

In [26]:
df.to_parquet('../cleaned_data/clean_yellow_tripdata_2026-01.parquet', index=False)

In [30]:
df.shape

(3618604, 22)

# vendor id 7 is get completely removed - means it has only dirty data

In [28]:
df['vendor_id'].value_counts()

vendor_id
2    2915371
1     699230
6       4003
Name: count, dtype: int64

# function to clean pickup and dropoff datetime

In [29]:
def clean_pickup_dropoff_datetime(df):
    df.rename(columns={
        'tpep_pickup_datetime' : 'pickup_datetime',
        'tpep_dropoff_datetime' : 'dropoff_datetime'
    }, inplace=True)

    df = df[(df['pickup_datetime'] < df['dropoff_datetime'])]

    df = df[
        (df['pickup_datetime'] >= '2025-12-31') |
        (df['dropoff_datetime'] <= '2026-02-01')
    ]

    df['trip_duration'] = df['dropoff_datetime'] - df['pickup_datetime']

    df = df[(df['trip_duration'] >= pd.Timedelta(minutes=2))]